In [ ]:
import pandas as pd

matches = pd.read_csv(r"/content/matches (1).csv")
deliveries = pd.read_csv(r"/content/deliveries (2).csv")

matches.head()
deliveries.head()

,match_id,inning,batting_team,bowling_team,over,ball,batsman,non_striker,bowler,is_super_over,...,bye_runs,legbye_runs,noball_runs,penalty_runs,batsman_runs,extra_runs,total_runs,player_dismissed,dismissal_kind,fielder
0,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,1,DA Warner,S Dhawan,TS Mills,0,...,0,0,0,0,0,0,0,NaN,NaN,NaN
1,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,2,DA Warner,S Dhawan,TS Mills,0,...,0,0,0,0,0,0,0,NaN,NaN,NaN
2,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,3,DA Warner,S Dhawan,TS Mills,0,...,0,0,0,0,4,0,4,NaN,NaN,NaN
3,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,4,DA Warner,S Dhawan,TS Mills,0,...,0,0,0,0,0,0,0,NaN,NaN,NaN
4,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,5,DA Warner,S Dhawan,TS Mills,0,...,0,0,0,0,0,2,2,NaN,NaN,NaN


In [ ]:
deliveries.columns

Index(['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball',
       'batsman', 'non_striker', 'bowler', 'is_super_over', 'wide_runs',
       'bye_runs', 'legbye_runs', 'noball_runs', 'penalty_runs',
       'batsman_runs', 'extra_runs', 'total_runs', 'player_dismissed',
       'dismissal_kind', 'fielder'],
      dtype='object')

In [ ]:
pp_data = deliveries[deliveries['over'] <= 6]

In [ ]:
deliveries['is_wicket'] = deliveries['player_dismissed'].notnull().astype(int)

pp_data = deliveries[deliveries['over'] <= 6]

pp_summary = pp_data.groupby(['match_id', 'batting_team']).agg({
    'total_runs': 'sum',
    'is_wicket': 'sum'
}).reset_index()

In [ ]:
pp_summary.rename(columns={
    'total_runs': 'current_score',
    'is_wicket': 'wickets_lost'
}, inplace=True)

In [ ]:
df = pp_summary.merge(
    matches[['id', 'winner']],
    left_on='match_id',
    right_on='id'
)

In [ ]:
df['win'] = (df['batting_team'] == df['winner']).astype(int)

In [ ]:
df = df[['batting_team', 'current_score', 'wickets_lost', 'win']]

df.head()

,batting_team,current_score,wickets_lost,win
0,Royal Challengers Bangalore,54,1,0
1,Sunrisers Hyderabad,59,1,1
2,Mumbai Indians,61,1,0
3,Rising Pune Supergiant,59,1,1
4,Gujarat Lions,52,1,0


In [ ]:
df['run_rate'] = df['current_score'] / 6

In [ ]:
df['pressure'] = df['current_score'] / (df['wickets_lost'] + 1)

In [ ]:
df.columns

Index(['current_score', 'wickets_lost', 'win', 'run_rate', 'pressure',
       'batting_team_Chennai Super Kings', 'batting_team_Deccan Chargers',
       'batting_team_Delhi Daredevils', 'batting_team_Gujarat Lions',
       'batting_team_Kings XI Punjab', 'batting_team_Kochi Tuskers Kerala',
       'batting_team_Kolkata Knight Riders', 'batting_team_Mumbai Indians',
       'batting_team_Pune Warriors', 'batting_team_Rajasthan Royals',
       'batting_team_Rising Pune Supergiant',
       'batting_team_Rising Pune Supergiants',
       'batting_team_Royal Challengers Bangalore',
       'batting_team_Sunrisers Hyderabad'],
      dtype='object')

In [ ]:
X = df.drop('win', axis=1)
y = df['win']

In [ ]:
print(X.shape)
print(y.shape)

(1270, 18)
(1270,)


In [ ]:
df.head()

,current_score,wickets_lost,win,run_rate,pressure,batting_team_Chennai Super Kings,batting_team_Deccan Chargers,batting_team_Delhi Daredevils,batting_team_Gujarat Lions,batting_team_Kings XI Punjab,batting_team_Kochi Tuskers Kerala,batting_team_Kolkata Knight Riders,batting_team_Mumbai Indians,batting_team_Pune Warriors,batting_team_Rajasthan Royals,batting_team_Rising Pune Supergiant,batting_team_Rising Pune Supergiants,batting_team_Royal Challengers Bangalore,batting_team_Sunrisers Hyderabad
0,54,1,0,9.000000,27.0,False,False,False,False,False,False,False,False,False,False,False,False,True,False
1,59,1,1,9.833333,29.5,False,False,False,False,False,False,False,False,False,False,False,False,False,True
2,61,1,0,10.166667,30.5,False,False,False,False,False,False,False,True,False,False,False,False,False,False
3,59,1,1,9.833333,29.5,False,False,False,False,False,False,False,False,False,False,True,False,False,False
4,52,1,0,8.666667,26.0,False,False,False,True,False,False,False,False,False,False,False,False,False,False


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [ ]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.6771653543307087


In [ ]:
model.predict_proba([[60, 1, 10, 30] + [0]*len(X.columns[4:])])

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


array([[0.3833302, 0.6166698]])

In [ ]:
import pandas as pd

sample = pd.DataFrame([X.iloc[0].values], columns=X.columns)
model.predict_proba(sample)

array([[0.41973603, 0.58026397]])

In [ ]:
import pandas as pd

importance = pd.Series(model.coef_[0], index=X.columns)
importance.sort_values(ascending=False).head(10)

,0
batting_team_Mumbai Indians,0.532820
batting_team_Rising Pune Supergiant,0.450600
batting_team_Chennai Super Kings,0.419388
batting_team_Rajasthan Royals,0.308570
batting_team_Kolkata Knight Riders,0.240562
batting_team_Sunrisers Hyderabad,0.070590
current_score,0.022489
run_rate,0.003748
pressure,0.003512
batting_team_Royal Challengers Bangalore,-0.002360


In [ ]:
import plotly.express as px

fig = px.histogram(df, x='current_score', color='win',
                   title="Score vs Win Distribution")

fig.show()

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()
model.fit(X_train, y_train)

RandomForestClassifier()

In [ ]:
import pickle

pickle.dump(model, open("ipl_model.pkl", "wb"))